Watchtower keeps teaching material, operational code, and historical source in one repository. A notebook's existence alone does not say whether it is current, published, or relevant to the chapter an agent is editing. The active catalog identifies logical works; the CLI provides their metadata, plans, source cells, and publication operations.

This guide follows the current `content/` model and the revision-aware CLI. It traces a command from the catalog to a notebook cell, explains the roles of course contracts and generated pages, and shows how Kanban cards can organize an agent's work. The running example is a hypothetical `signals` course: revise its filtering chapter, inspect the result, and hand the work back for review without loading the entire course.

Run every command from the repository root through `.venv/bin/wt`. The virtual environment is required. Start with `.venv/bin/wt map`, then discover arguments with `.venv/bin/wt --help` and the relevant command's `--help`. Agents use the CLI; Watchtower has no MCP server.

## Repository map: authored content and generated views

The active registry is `backend/data/catalog.yaml`. It names posts, courses, chapters, portfolio entries, executable projects, personal notebooks, and the built-in photo gallery. Authored notebooks and their sidecars live under `content/notebooks/`; managed image uploads live under `backend/assets/`. Context files linked to artifacts or cards live under `backend/attachments/` and stay off the site. Active code lives under `projects/`.

The registry describes logical works rather than every file they use. An image, test, or project module belongs to a work without needing its own catalog entry. Operational code in `src/watchtower/` and `scripts/` also stays outside the catalog.

| Question | Source of truth |
|:--|:--|
| Is this a registered current work? | `backend/data/catalog.yaml` |
| What does a course intend, what is checked, and what is its chapter order? | `backend/data/courses/<slug>.yaml` |
| What should a learner read first? | The course's `index.ipynb` |
| How is a subject taught? | The authored post or chapter notebook |
| What does a portfolio entry contain, and where is its code? | `backend/data/portfolio.yaml` and its registered project |
| What supplies Home and Résumé? | `backend/data/profile.yaml` |
| What supplies Personal? | Ordered photo records in `backend/data/photos.yaml` |
| What work is assigned for the author? | `backend/data/kanban.yaml` |
| How is content presented? | Settings, templates, and site assets under `frontend/`, plus stored notebook outputs |

Personal is rendered once from a template and the photo records. Each photo has a heading, caption, lifecycle, optional image path, and optional width. A draft may show a “No Photo” placeholder; a published photo must point to an existing image. There is no generated gallery notebook to author and no collection-level publication switch.

`frontend/generated/` contains disposable Quarto projects and rendered sites. `backend/runtime/` contains locks, build records, and transaction journals. Neither directory is an authored source. Generated headers, listings, résumé outputs, and course-context includes are changed through their inputs and regeneration commands.

Historical work remains under `archive/2026-09-30/`; retired publishing inputs live under `archive/2026-10-01/content-system-inputs/`. Deletions have their own `archive/deleted/` records. Normal discovery follows the active catalog; deliberate archive searches expose historical source without treating it as a current fact.

`content/` is the authoring workspace: edit notebook bodies and sidecars in VS Code or Jupyter. Use the CMS, CLI or API for managed metadata, uploads and context files under `backend/`. Register new notebooks and retire active entries through those interfaces so links and metadata remain consistent. Validation and rendering happen before successful output is promoted, so a failed authoring edit leaves the last successful site available.

Deleted history under `archive/deleted/` is disposable. Manually removing files or folders there does not affect active records, workspace revisions or builds. Permanent ID/source reservations remain in the catalog; historical project directories referenced by active portfolio entries remain source dependencies.

## The catalog carries identity and publication state

A stable ID is the handle for a work. A chapter record can look like this:

```yaml
- id: course/signals/02-filtering
  kind: chapter
  title: Filtering
  toc_title: 02. Filtering
  path: content/notebooks/courses/signals/02-filtering.ipynb
  parent: course/signals
  section: fundamentals
  visibility: public
  lifecycle: draft
  relations: []
  planned: {}
```

The course record points to its directory; the chapter points to its notebook. `parent` and `section` tie the chapter to its course contract. `toc_title` is a short navigation label, independent of the full title. An authored chapter has one real H1 matching that full title. `relations` contains exact registered stable IDs. Courses and chapters use their ordered outline instead of tags; creation and metadata updates reject tags for these kinds. Other catalog kinds use tags as their single taxonomy, projected as categories in the generated site. Notebook cell tags remain available for cell selection.

A planned record can reserve a source path before a notebook exists. Portfolio notebook and code details live in portfolio YAML rather than duplicated catalog paths. The generated site's route can retain a legacy `nb/...` URL even though its canonical source has moved under `content/notebooks/`. A URL is therefore not a reliable filesystem locator.

`visibility` is `public` or `private`; `lifecycle` is `planned`, `draft`, or `published`. Planned entries have no authored body. Draft entries require a source notebook and may contain only a title; Published requires authored content. The working preview shows valid authoring states, including draft diagnostic panels. Production includes only public published pages. Planned entries never render in preview or production. Public Portfolio cards require published entries.

Course eligibility also applies to descendants. A private or draft course suppresses its children from production, including their links and assets, while preserving their individual states. A planned course suppresses all of its children in both modes. Publishing a chapter requires a public published parent.

`map`, `ls`, and default `find` use the active registry. `map` groups courses into home, optional overview, and chapter paths, but it does not expose full chapter metadata or the authoritative teaching order. Read the contract's ordered TOC for that order. `find` searches registered sources case-insensitively; notebook hits report `[cell N]`, while ordinary text files report line numbers.

The catalog governs discovery rather than every direct notebook read. `cat` can resolve an explicit existing notebook path, including an archive path, without making it active. A successful read does not certify registration. Use `validate` to detect unregistered sources and inconsistent records.

## Course contracts keep intent distinct from evidence

A course contract gives concise purpose and audience, separate planned and actualized facts, and an ordered TOC. Chapter plans are stored alongside that order:

```yaml
id: course/signals
purpose: Help readers reason about sampled signals.
audience: Readers comfortable with Python and basic algebra.
planned:
  summary: Build a filter after introducing sampling and noise.
  chapters:
    - chapter_id: course/signals/02-filtering
      section: fundamentals
      summary: Compare a moving average with the unfiltered signal.
      content: Explain averaging and its effect on short fluctuations.
      lab_and_evidence: Plot both signals and check the window-size tradeoff.
actualized:
  summary: The sampling lesson and its exercises have been checked.
toc:
  - id: fundamentals
    title: Fundamentals
    chapters: [course/signals/02-filtering]
```

`planned` describes the intended work. `actualized` records completed, checked work explicitly. A file, draft, sidebar entry, or Kanban move is not completion evidence. Validation can check structure and references, but it cannot establish that an educational claim is true.

The home, `index.ipynb`, is the learner-facing course README. It explains prerequisites, progression, and how to work through the lessons. An optional `00-overview.ipynb` develops the whole-course technical design. Chapters teach local explanations, examples, and exercises. The repository `README.md` explains Watchtower itself.

Course homes and the CMS derive the chapter table from the ordered TOC, full catalog titles, and chapter-plan summaries. Do not maintain a second table in the notebook. Shared facts and navigation belong in YAML; teaching belongs in authored cells.

Build generation supplies the concise course-context include inside the generated Quarto project. It also supports older include directives in course homes. The authored notebook does not need a manually regenerated sidecar. `make knowledge`, `wt render-context`, and `wt sync-site` regenerate compatible projections from saved inputs.

Planning updates follow the owning record. `update <id> --summary` changes its public sentence; `--internal-notes` changes CMS-only context. `--plan-file` accepts Markdown under `.tmp/`, with optional `##` sections named by that kind's planning labels or keys. `wt plan <stable-id>` lists the effective fields and saved build brief. Existing optional planning fields and actualized facts are preserved.

## Gather context in the order the task needs it

For an authored chapter, begin with:

```sh
.venv/bin/wt context course/signals/02-filtering
.venv/bin/wt data course/signals
.venv/bin/wt plan course/signals/02-filtering
```

The hypothetical IDs here must be registered before these commands can be used. `context` returns the catalog record, parsed course contract, and reading paths for the home, optional overview, and target. It does not load all chapter cells. `data` exposes the saved contract and its workspace revision for a coordinated metadata save. `plan` reads saved authoring intent for an exact registered stable ID, in any lifecycle. Its JSON includes the public summary, effective planning fields and their definitions, internal notes, attachments, saved build brief, source path, and workspace revision. A chapter brief includes parent-course context; a course brief includes its chapter plans in TOC order.

Read those layers in this order:

```text
catalog record + backend/data/courses/signals.yaml
                       ↓
index.ipynb: learner route and prerequisites
                       ↓
00-overview.ipynb, if present: whole-course technical design
                       ↓
02-filtering.ipynb: the requested work
                       ↓
adjacent chapter openings, only for a relevant handoff
```

A changed exercise sequence may require the preceding and following introductions. Read them to answer a concrete dependency question. The default should not be an automatic sweep of all siblings. For a planned chapter without a source notebook, inspect its saved plan before using `start` to materialize a draft.

The CLI has a second meaning of context: `cat <notebook> --index 4 --context 2` prints cells 2 through 6 of one notebook, marking surrounding cells as context. This supplies local coherence for an edit; it does not consult sibling chapters.

The current `context` command exposes less than the service's full inspection response and does not return a revision. Use `wt plan <stable-id>` to retrieve the saved build brief; the CMS editor no longer displays a separate build-brief panel. Neither read certifies readiness to publish. Missing core-field guidance concerns the saved plan, so an authored notebook can be complete even when its old plan is partial or empty. Read its current cells with `wt cat` and check the work itself.

Historical research is explicit: use `.venv/bin/wt map --archive`, `.venv/bin/wt ls courses --archive`, or `.venv/bin/wt find "filter" --archive`. An archive hit is prior source to assess, not evidence about the active course.

## Follow a command to the cell layer

`cli.py` defines Typer arguments and imports command modules on demand. `content_cli.py` assembles structured content commands, and `kanban_cli.py` assembles the task commands. Content, Kanban, and build services are shared with the HTTP API and CMS. `knowledge.py` and `inspect.py` supply discovery and compatibility helpers. Both console scripts, `wt` and `watchtower`, enter `watchtower.cli:main`.

A chapter read follows this path:

```text
.venv/bin/wt cat course/signals/02-filtering --index 3 --context 1
  → Typer parses the options
  → the managed workspace gate protects the operation
  → notebook.cat_notebook resolves the name
  → inspect.resolve_ipynb consults catalog IDs and path forms
  → nbformat parses the notebook
  → selected cells are rendered as Markdown-like text
```

Notebook resolution accepts registered IDs, relative paths with or without `.ipynb`, tier-prefixed paths, and convenient stems. Use the full ID or path when several chapters share a stem. A course ID resolves to its home. `cat` selects zero-based indices, half-open ranges such as `2:5`, Jupyter tags, or leading Quarto `#| label:` options. A tag can match several cells.

Source reads default to 4,096 characters per cell. `--offset` and `--limit` page through long text; `--limit 0` removes the source limit. `--with-outputs` adds stored outputs while summarizing image payloads. `output --index N` extracts images to `.tmp/` for visual inspection rather than printing base64.

`edit-cell` changes one cell's source and preserves outputs and metadata. `append-cell`, `insert-cell`, `remove-cell`, `tag`, and `clear-outputs` cover other changes through `nbformat`. Editing and tagging require unique matches; removal can intentionally remove all cells matching a tag. Source writes are capped at 20,000 characters per cell. Re-read after structural changes because indices move.

For a guarded edit, first read with `cat --with-revision`. Its header contains a `notebook:<hash>` token covering the exact notebook bytes, including outputs and metadata. Supply that token with `--expected-revision` on the cell mutation. A stale token rejects the edit before it can change a different cell or replace another agent's work. Leaving out the option retains ordinary single-agent editing behavior.

## Plan, start, execute, and publish explicitly

Creation registers a private plan; `start` creates its private draft notebook. For example:

```sh
mkdir -p .tmp
cat > .tmp/example-plan.md <<'PLAN'
## Outline

Explain the question and check an example.
PLAN
.venv/bin/wt new post example --title "Example" \
  --plan-file .tmp/example-plan.md
.venv/bin/wt plan post/example
.venv/bin/wt start post/example
.venv/bin/wt cat post/example --with-revision
# Author the body through cell operations, carrying the notebook token.
.venv/bin/wt diff post/example
.venv/bin/wt validate
make preview PORT=4300
# After reviewing the authored result:
.venv/bin/wt publish post/example
make build
```

`new course` registers a contract and planned course home. It does not create chapter notebooks or an authored first chapter. A title is enough to start; the purpose, audience and summary can be added later. `new chapter` registers a chapter plan and TOC entry; partial and empty plans can also be started. `new section` adds an outline section. The CMS course workspace provides section and chapter reordering, while the CLI also supports structured contract updates.

Larger plans use files under `.tmp/`. Every kind uses optional `##` sections named by the planning labels or keys returned by `wt plan`. An empty section clears that field; omitted sections retain saved values. Scope context belongs in the existing plan fields or Internal notes, and next steps belong in linked Kanban cards. The saved plan survives removal of the scratch file. Starting an active portfolio also initializes and registers its configured project, reusing existing code without overwriting it.

`start` seeds an editable notebook section for each filled plan field. Public summaries and internal notes are not seeded into the body. While a draft still matches its last seed, plan saves refresh those sections and can fill a still-blank portfolio abstract. The first authored edit to its section content stops automatic reseeding; subsequent plan saves preserve authored cells and explicit abstracts.

Saved plans (`wt plan <stable-id>`) capture initial authoring intent and may become stale as drafts develop. Before continuing, reviewing, or summarizing started content, read the current notebook with `wt cat`. Purpose, audience, and scope guide the work; outlines, examples, and technical approaches can evolve as understanding improves. Explain discoveries that justify changes, and flag material expansions or redirections before proceeding unless already authorized. Once agreed, update only the relevant plan fields. Routine drafting does not require plan synchronization. Keep public summaries and course navigation accurate.

`import` normalizes an external notebook, preserves stored outputs, and defaults authored content to draft. `register` can register existing project code. Metadata changes use `update`; structured records use `data` or `gallery`; related repairs can use `batch --file .tmp/batch.json`. Publication checks include portfolio abstract, image, and caption requirements, and chapter parent eligibility.

`run` is the explicit execution path. It uses the notebook's kernelspec, falling back to `python3`. With `--index N`, it runs the prefix through N in a fresh kernel so earlier imports and variables are available, then writes back only N's outputs and execution count. Cell errors are stored inline and cause exit 1. Rendering uses stored outputs without executing code.

Execution captures the notebook under the workspace lock and releases the lock while the kernel runs. Before saving, it checks that the source is unchanged. If another agent changed or deleted it, the command reports a conflict and retains results under `.tmp/execution-conflicts/` for review. Kanban reads and updates can proceed during the run.

`publish` atomically sets the lifecycle to Published and visibility to public, changing eligibility for the next deployment. It does not commit, push, or demonstrate that deployment happened. `draft` preserves the body and visibility while withdrawing published content. Builds promote only successful output; GitHub Actions rebuilds committed inputs on `main` and deploys to the existing `gh-pages` target.

Removal also has an explicit review. `delete <id> --dry-run` returns the affected entries, links, and revision; apply that revision with `delete`, adding `--cascade` for a course's chapters. Notebook sources and course contracts are archived byte-for-byte. Images and project code remain, linked records are detached, and removed IDs and source names stay reserved.

## Revisions protect different decisions

A revision is a token for the state a caller used to decide on a write. Its scope matters:

| Token | Covers | Used by |
|:--|:--|:--|
| Workspace `revision` | Exact managed content and site-input bytes, plus project directory identity | Metadata, structured-data, lifecycle and deletion operations; also accepted by Kanban |
| `notebook:<hash>` | One notebook's exact bytes | Cell mutations and execution through `--expected-revision` |
| Kanban `board_revision`, formatted `kanban:<hash>` | The saved board's bytes | Kanban card mutations through `--expected-revision` or HTTP `If-Match` |

`data`, `gallery`, and `kanban ls` expose workspace revisions. Kanban additionally exposes its board token. A task-only update can use that narrower token so an unrelated notebook edit does not invalidate the decision. If the decision depends on content as well as the task, use the workspace revision. A changed card still invalidates the shared board token; there is no per-card revision yet.

The API requires `If-Match` for writes, and CMS forms carry revisions. CLI options retain compatibility with single-agent calls that omit a token, capturing current inputs during the operation. That does not protect a decision based on an earlier read. Carry an explicit token when coordinating agents.

An explicit stale revision is rejected rather than retried automatically. The console reports service conflicts with a nonzero exit status; notebook execution errors and validation failures also return failure. Read the error payload or message, inspect current state, and make a new decision before trying again.

Managed clients share a lock and durable transaction journals under `backend/runtime/`. A write validates its proposed final state before installing it. Recovery rolls forward from matching preimages or expected absence. External edits block recovery while the competing versions are retained; a conflicted journal also prevents a build from replacing successful output. Never reset files blindly to unblock it.

The lock coordinates Watchtower clients. Uncoordinated IDE saves retain a small check-to-replace race, so pause affected saves when strict coordination is needed. Project code bytes are not included in workspace revisions: a task token or workspace token is not proof that a project's implementation passed its checks.

## Kanban records author work without publishing it

Kanban has four columns: To do, In progress, Review, and Done. Their CLI values are `todo`, `in-progress`, `review`, and `done`. The board is stored in `backend/data/kanban.yaml`; an absent file means an empty board. It stays off public pages, and task-only changes do not trigger automatic site rebuilds.

Each card has an immutable internal ID, a human reference such as `card#12`, title, description, column, and ordered `artifact_ids`. A representative saved board is:

```yaml
version: 1
next_number: 13
cards:
  - id: review-filtering
    ref: card#12
    title: Review the filtering exercise
    description: Check the window-size comparison and record the result.
    column: todo
    artifact_ids: [course/signals/02-filtering]
```

References are assigned by card creation and work with `update`, `move`, and `rm`. Numbers survive ordinary removal and whole-board replacement. Whole-board data and batch saves preserve existing references and the numbering counter; a reused reference or a reassignment to a different ID is rejected. Explicit repair of malformed data requires care to retain the known counter and identities.

Start by reading the board:

```sh
.venv/bin/wt kanban ls
.venv/bin/wt kanban ls --column review
.venv/bin/wt kanban ls --query "card#12"
```

The query searches internal IDs, human references, title, description, and linked artifact IDs. `wt kanban show card#12` returns the card and current linked plans, internal notes, attachments and parent-course context. Context files can be attached to a card or its artifacts; `wt attachments ls <owner>` returns their ownership and file paths. Copy the returned `board_revision` into a shell variable such as `task_revision` before a task-only save:

```sh
.venv/bin/wt kanban add --title "Review the CLI article" \
  --description "Check examples against command help and inspect the rendered page." \
  --link post/001-watchtower-cli-and-knowledge-model \
  --expected-revision "$task_revision"
```

Use the returned card reference and the new returned token for the next update. `--link` can be repeated, but links must be exact registered artifact IDs. A title, notebook path, or free-form search string cannot stand in for an ID. `update --link` replaces the link list; `--clear-links` removes it, and the two options cannot be combined.

Card reads include artifact titles, source paths, CMS URLs, working-preview URLs, and VS Code links when canonical sources exist. Planned notebooks have no editor link until started; project directories and gallery data use their actual sources. A card link provides navigation, not evidence that the linked work is complete.

Moving a card to Review or Done changes only the task. It does not start a notebook, publish an artifact, update a course's actualized facts, or deploy the site.

## Coordinate agents through one authoritative board

Moving a card into In progress does not reserve it. Two workers can both perform that move successfully after separate fresh reads. The current card model has no owner, dependency graph, atomic claim, or structured result field. Descriptions can record those facts as prose, but the CLI does not enforce them.

For the current implementation, let one coordinator write the board and assign work explicitly:

1. Create a card for a concrete deliverable and link its registered artifact. Put the assigned worker, file scope, prerequisites, acceptance checks, and expected handoff in its description.
2. Read artifact context and confirm the source and lifecycle. Assign one worker per notebook; give it the relevant contract and cell reading paths.
3. Read the latest board token, move the assigned card to In progress, and dispatch the worker through the host's agent tools.
4. Have the worker use notebook tokens for edits, run relevant checks, and return changed paths, observations, and evidence to the coordinator.
5. Consolidate that result into the description and move the card to Review. Review the content and rendered result before marking the task Done or publishing an artifact.

For a returned card reference in `task_ref`, the review transition is:

```sh
.venv/bin/wt kanban ls --query "$task_ref"
# Refresh task_revision from this response's board_revision.
.venv/bin/wt kanban move "$task_ref" review --expected-revision "$task_revision"
```

Do not reuse the token from before an intervening card update. A conflict requires another read and a deliberate decision, not a blind retry. Likewise, updating a description replaces it; concurrent workers should return their notes to the coordinator rather than independently overwrite the handoff.

Worktrees isolate content and code changes, but each checkout also has its own saved board and runtime lock. Choose one coordinator checkout as authoritative for assignments and progress. Workers use their worktrees for source changes and report centrally; merging independently edited Kanban YAML is not an ownership protocol.

This workflow provides reviewable task records and avoids duplicate assignments through the coordinator. Atomic claims, machine-checked dependencies, and structured handoffs would make that coordination enforceable, but they are separate future capabilities. The CLI records work; the host launches agents.

## Tests protect the connections between these layers

Tests use isolated workspaces so active content, archives, and real card state do not affect assertions. The suite includes CLI boundaries, domain services, HTTP/CMS interactions, generated-site checks, and actual Jupyter execution. Older navigation fixtures exercise the retained pre-migration compatibility paths; current content-service fixtures exercise `backend/data/` and `content/notebooks/`.

| Behavior | Focused evidence |
|:--|:--|
| Discovery and notebook resolution | `test_inspect.py`, `test_knowledge.py`, and `test_cli_catalog.py` exercise maps, search, paths, IDs, and explicit archives. |
| Plans and readiness | `test_cli_content.py`, `test_content_service.py`, and `test_cms_planning.py` cover saved plans, start requirements, contract/TOC organization, and preservation of optional fields. |
| Publication and generation | `test_build.py` and `test_domain_acceptance.py` cover placeholders, authored notebook copying, parent filtering, outputs/assets, and successful-build promotion. |
| Cell operations and execution | `test_notebook.py`, `test_outputs.py`, and `test_execute.py` exercise locators, limits, output extraction, prefix state, and errors. |
| Kanban transport parity | `test_kanban.py` checks shared CLI/API/CMS state, validated links, stale revisions, publication independence, and rebuild behavior. |
| Agent concurrency regressions | `test_cli_agent_safety.py` covers guarded cell writes, execution without a long workspace lock, retained conflicting results, scoped board revisions, effective plan updates, and card identity preservation. |
| Deletion and recovery | `test_deletion.py` and `test_content_service.py` check retained source, detached links, reserved names, crash recovery, and external-edit conflicts. |

A helper test can prove that source search ignores an orphan; a command test proves that `find` actually reaches that behavior. Transport tests check the same service through different clients. Neither layer replaces the other.

One agent regression begins with a notebook token, inserts a cell through a second operation, and attempts a stale positional edit. The source must remain unchanged. Another starts execution while a separate thread reads and moves a Kanban card; it must complete before execution saves its outputs. Execution-conflict tests change or delete the source and then inspect retained results. Whole-board tests clear cards while keeping the numbering history and reject reassigned references.

These scenarios prove concrete mechanical boundaries. They do not prove that a lesson teaches well, that an actualized summary is honest, or that a Done card satisfies the reviewer. Those judgments still require evidence and review.

## Validate the source and the reader-facing result

Review notebook changes with `.venv/bin/wt diff <id>`. A moved notebook can use `--base-source <old-repository-path>`. For edited code, run the relevant cell with `run --index N` and inspect its stored output. A Markdown-only article needs no kernel run.

`validate` checks schemas, stable references, lifecycle/content agreement, chapter H1s, ordered course membership, required assets, and unregistered active files. Content mutations validate their proposed final state, so an old lifecycle disagreement can be repaired without first making the old state pass validation. Malformed YAML and duplicate keys still require explicit repair.

For Python behavior changes, use `make lint`, `make typecheck`, and `make test`. For content, validate and render the affected pages with the working preview or build. `make preview PORT=4300` watches saved inputs and serves successful working output; `make build` renders production eligibility. Both use stored outputs. A failed build keeps the last successful result.

Inspect the rendered page as well as the source: code examples, tables, callouts, navigation, and publication panels can be correct text yet still render poorly. Keep secrets in the OS keyring through `wt vault`, and never print or commit their values. When the CLI surface changes, update `README.md` and `AGENTS.md` with it so the next agent can follow an executable workflow.